# CO2-Aware Workload Migration: Where Should SURF Run Its Workload?

*Use case 3 of 3, about 10 minutes.*

We envision digital twins becoming primary instruments in datacenter operations, continuously and autonomously helping
with major operational decisions and with adapting ICT infrastructure, live, with a human-in-the-loop.
Here, OpenDT is an advisor-on-the-side for workload migration, CO2 aware: the same SURF workload, on the same hosts, on
the grids of the Netherlands (NL), Germany (DE), Romania (RO), and Poland (PL).

In [ ]:
import pandas as pd
from resources import *

check_setup()

## 1. Carbon Intensity of Four Grids

Carbon intensity [gCO2/kWh] from Electricity Maps (October 2022, hourly, ODbL; see `resources/carbon_traces/LICENSE.md`).

*Question:* SURF runs in NL; which of the four grids has the lowest carbon intensity during the SURF week (6-13 October
2022)?

In [ ]:
week = {zone: carbon_intensity(zone)[TRACE_START : TRACE_END - pd.Timedelta("1h")] for zone in CARBON_ZONES}
intensity = pd.DataFrame(week)  # 168 hourly values per grid

ax = intensity.plot()
ax.set(xlabel="Time [UTC]", ylabel="Carbon Intensity [gCO2/kWh]", ylim=(0, None))
ax.yaxis.set_major_formatter("{x:,.0f}")  # 1,000
legend_above(ax)
pd.DataFrame({"Average [gCO2/kWh]": intensity.mean()}).round()

*Question:* is the grid with the lowest average also the greenest in every one of the 168 hours?

<details><summary>Answer</summary>

No. RO has the lowest average (343 gCO2/kWh), but NL is greener in 57 of the 168 hours; the carbon intensity
of NL changes more over the week (from 123 to 528 gCO2/kWh).
</details>

## 2. One Simulation per Grid

Only the carbon intensity trace changes between the simulations (`powerSource.carbonTracePath` in the topology).

*Question:* between the four simulations, which numbers change (energy, CO2, or both), and why?

In [ ]:
runs = simulate_many("carbon", CARBON_ZONES, until=TRACE_END)  # one simulation of the full week per grid

table = pd.DataFrame([run.summary() for run in runs.values()], index=list(runs))
table[["energy_kwh", "co2_kg"]].round()

<details><summary>Answer</summary>

The same energy on every grid (3,424 kWh), since the workload and the hosts are the same; the CO2 emissions go
from 1,172 kg (RO) to 2,959 kg (PL), with the carbon intensity of the grid.
</details>

In [ ]:
co2 = table["co2_kg"]
ax = co2.sort_values(ascending=False).plot.barh(figsize=(8, 2.5))
ax.set(xlabel="CO2 Emissions [kg]")
ax.xaxis.set_major_formatter("{x:,.0f}")

best = co2.idxmin()
saving = co2["NL"] - co2[best]
print(f"Advice: run the SURF workload in {best}, with {saving:,.0f} kg CO2 less per week than in NL "
      f"({saving / co2['NL']:.0%}).")

## 3. The Human-in-the-Loop

![Datacenter digital twinning](resources/images/digital_twinning.png)

*High-level overview of datacenter digital twinning; a human-in-the-loop (HITL) oversees the process (OpenDT paper,
Figure 1).*

*Question:* should SURF migrate its workload to RO, and what else does the human-in-the-loop weigh before such a
decision?

<details><summary>Answer</summary>

The saving is small (6% vs. NL), and NL is greener than RO in a third of the hours.
Before migrating, the human-in-the-loop considers, e.g., (1) the energy and time to move data and jobs over the
network, (2) the latency for the users, (3) the rules on where data may be stored, (4) the free capacity and the cost
of the remote datacenter, and (5) the CO2 emitted to build new hardware.
</details>

*Exercise 1:* with a workload that follows the greenest of the four grids, hour by hour (next cell), how much CO2 does
SURF emit in the week, and how many migrations does this take?

In [ ]:
emissions = pd.DataFrame({zone: run.power["carbon_emission"] for zone, run in runs.items()})  # [g]
co2_per_hour = emissions.resample("1h").sum() / 1000  # [kg]
greenest = co2_per_hour.idxmin(axis=1)  # the greenest grid in each hour
switches = (greenest != greenest.shift())[1:].sum()  # migrations from one hour to the next
print(f"Greenest grid every hour: {co2_per_hour.min(axis=1).sum():,.0f} kg CO2, with {switches} migrations")

<details><summary>Answer</summary>

1,065 kg, 15% less than in NL and 9% less than in RO; however, the workload migrates 11 times in one week,
between NL and RO, and each migration costs energy and time.
</details>

*Exercise 2 (optional):* combining notebooks 1 and 3, SURF could run in RO with 1/2 of the hosts (138); compare its CO2
and waiting jobs with NL and all 277 hosts.

In [ ]:
half = simulate(until=TRACE_END, hosts=138, carbon="RO")
pd.DataFrame([runs["NL"].summary(), half.summary()], index=["NL, 277 hosts", "RO, 138 hosts"]).round(1)

<details><summary>Answer</summary>

971 kg CO2, 23% less than NL with 277 hosts; however, 2,631 jobs wait for a free host, 78 minutes per job on
average.
</details>

*Next (optional):* [4.live_twin.ipynb](4.live_twin.ipynb), the live digital twin.